# 도착 전 잔여좌석 모델 학습 — Standalone Colab

이 파일은 필요한 프로젝트 소스를 내부에 포함합니다. GBIS API와 Colab Secrets 외부 파일을 참조하지 않고, 주 모델과 4× 전환 가중 모델을 정확한 전체 데이터로 학습합니다.

HGB/ExtraTrees/LightGBM 3개 모델을 두 번 학습하므로 CPU High-RAM 기준 수 시간이 걸릴 수 있습니다. 샘플링·경량 대체는 수행하지 않습니다.

In [ ]:
# 이 셀은 노트북에 포함된 소스만 복원합니다. GitHub/로컬 저장소를 참조하지 않습니다.
API_BASE_URL = "https://161.33.212.6"
DRIVE_CACHE_PATH = "/content/drive/MyDrive/GBIS/gbis_api_cache.sqlite3"

import base64
import importlib
import importlib.abc
import importlib.util
import io
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "joblib>=1.4,<2", "numpy>=2.0,<3", "pandas>=2.2,<4",
    "scikit-learn>=1.5,<2", "lightgbm>=4.5,<5", "httpx>=0.27,<1",
], check=True)

SOURCE_BUNDLE_B64 = "..."
RUNTIME_ROOT = Path("/content/arrival_seat_standalone")
RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)

# Embedded project modules are loaded directly from this notebook's bytes.
# No repository, local Python file, or generated source file is referenced.
with zipfile.ZipFile(io.BytesIO(base64.b64decode(SOURCE_BUNDLE_B64))) as archive:
    EMBEDDED_MODULES = {name: archive.read(name) for name in archive.namelist()}

class EmbeddedModuleFinder(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def _path(self, fullname):
        top_level = fullname.replace(".", "/")
        candidates = []
        if fullname.startswith("gbis_client"):
            candidates = [f"{top_level}/__init__.py", f"{top_level}.py"]
        elif "." not in fullname:
            candidates = [f"analysis/{top_level}.py"]
        for candidate in candidates:
            if candidate in EMBEDDED_MODULES:
                return candidate
        return None

    def find_spec(self, fullname, path=None, target=None):
        filename = self._path(fullname)
        if filename is None:
            return None
        return importlib.util.spec_from_loader(
            fullname, self, is_package=filename.endswith("/__init__.py")
        )

    def create_module(self, spec):
        return None

    def exec_module(self, module):
        filename = self._path(module.__name__)
        module.__file__ = f"<embedded:{filename}>"
        if filename.endswith("/__init__.py"):
            module.__path__ = []
        exec(compile(EMBEDDED_MODULES[filename], module.__file__, "exec"), module.__dict__)

sys.meta_path.insert(0, EmbeddedModuleFinder())

# 이후 셀은 프로젝트 모듈을 import하지 않는다. 아래 단일 컨텍스트에 노출된
# 함수/상수만 사용한다. 이 모듈들은 모두 SOURCE_BUNDLE_B64에서만 읽힌다.
_gbis = importlib.import_module("gbis_client")
_route = importlib.import_module("route_specific_feature_experiment")
_pooled = importlib.import_module("pooled_main_model_overfit_ablation")
_hypothesis = importlib.import_module("hypothesis_model_search")
_registry = importlib.import_module("pooled_main_model_registry")
_primary_export = importlib.import_module("export_exact_primary_colab_model")
_weighted_export = importlib.import_module("export_exact_weighted_colab_model")
ENGINE = {
    "GBISApiCache": _gbis.GBISApiCache,
    "DEFAULT_ROUTES": _route.DEFAULT_ROUTES,
    "build_route_snapshots": _route.build_route_snapshots,
    "cache_paths": _route.cache_paths,
    "prepare_pooled_data": _pooled.prepare_pooled_data,
    "decode_target": _hypothesis.decode_target,
    "apply_pooled_feature_profile": _registry.apply_pooled_feature_profile,
    "export_primary": _primary_export.main,
    "export_weighted": _weighted_export.main,
}

from google.colab import drive, userdata
API_KEY = userdata.get("GBIS_API_KEY")
if not API_KEY:
    raise RuntimeError("Colab Secrets에 GBIS_API_KEY를 등록하고 Notebook access를 켜주세요.")
drive.mount("/content/drive")
LOCAL_CACHE = Path("/content/gbis_api_cache.sqlite3")
DRIVE_CACHE = Path(DRIVE_CACHE_PATH)
DRIVE_CACHE.parent.mkdir(parents=True, exist_ok=True)
CACHE_RESTORED = DRIVE_CACHE.is_file()
if CACHE_RESTORED:
    shutil.copy2(DRIVE_CACHE, LOCAL_CACHE)
    print("Google Drive 캐시를 복원했습니다.")
else:
    print("첫 실행: GBIS 전체 이력을 동기화합니다.")

TRAINING_CUTOFF_DATE = "2026-08-12"  # 완결된 학습일로 변경 가능


In [ ]:
# 원시 GBIS 데이터 동기화. 활성 6개 노선 전체를 수집합니다.
DEFAULT_ROUTES = ENGINE["DEFAULT_ROUTES"]
cache = ENGINE["GBISApiCache"](base_url=API_BASE_URL, api_key=API_KEY, cache_path=LOCAL_CACHE)
try:
    cache.refresh_routes()
    for index, route_id in enumerate(DEFAULT_ROUTES, 1):
        print(f"[sync {index}/6] {DEFAULT_ROUTES[route_id]} ({route_id})", flush=True)
        cache.refresh_stations(route_id)
        cache.refresh_full_history(route_id)
    cache.refresh_latest()
    routes_df = cache.routes_df()
    stations_df = cache.stations_df()
    latest_df = cache.latest_locations_df()
    history_df = cache.history_df()
finally:
    cache.close()
    if LOCAL_CACHE.is_file():
        shutil.copy2(LOCAL_CACHE, DRIVE_CACHE)

source_cutoff = str(history_df["observed_at"].max())
print(f"raw history rows: {len(history_df):,}; source cutoff: {source_cutoff}")


In [ ]:
# 원시 SQLite 이력 → strict-prior 통합 모델 피처
import json
import pandas as pd

feature_dir = RUNTIME_ROOT / "feature_cache"
for route_id in DEFAULT_ROUTES:
    print(f"[features] {DEFAULT_ROUTES[route_id]} ({route_id})", flush=True)
    snapshots, flows, metadata = ENGINE["build_route_snapshots"](
        LOCAL_CACHE, route_id, source_cutoff=source_cutoff
    )
    snapshot_path, flow_path, metadata_path = ENGINE["cache_paths"](feature_dir, route_id)
    feature_dir.mkdir(parents=True, exist_ok=True)
    snapshots.to_pickle(snapshot_path)
    flows.to_pickle(flow_path)
    metadata_path.write_text(json.dumps(metadata, ensure_ascii=False), encoding="utf-8")

pooled_features, route_metadata = ENGINE["prepare_pooled_data"](
    feature_dir, source_cutoff=source_cutoff
)
pooled_features.attrs["source_cutoff"] = source_cutoff
pooled_features.attrs["route_metadata"] = route_metadata
featured_cache = RUNTIME_ROOT / "pooled_features.pkl"
pooled_features.to_pickle(featured_cache)
print(f"prepared feature rows: {len(pooled_features):,}; events: {pooled_features.event_id.nunique():,}")


In [ ]:
# 정확한 전체 주 모델 export
import sys

primary_output = Path("/content/arrival_seat_primary_v1.pkl")
sys.argv = [
    "export_exact_primary_colab_model.py",
    "--featured-cache", str(featured_cache),
    "--cache-dir", str(feature_dir),
    "--output", str(primary_output),
    "--training-cutoff-date", TRAINING_CUTOFF_DATE,
    "--source-cutoff", source_cutoff,
]
if ENGINE["export_primary"]() != 0:
    raise RuntimeError("주 모델 export에 실패했습니다.")


In [ ]:
# 정확한 전체 4× 전환 가중 specialist export
weighted_output = Path("/content/arrival_seat_low25_weighted_v1.pkl")
sys.argv = [
    "export_exact_weighted_colab_model.py",
    "--featured-cache", str(featured_cache),
    "--cache-dir", str(feature_dir),
    "--output", str(weighted_output),
    "--training-cutoff-date", TRAINING_CUTOFF_DATE,
    "--source-cutoff", source_cutoff,
]
if ENGINE["export_weighted"]() != 0:
    raise RuntimeError("가중 모델 export에 실패했습니다.")


In [ ]:
# artifact 검증 및 다운로드
from google.colab import files
import joblib

for path in (primary_output, weighted_output):
    artifact = joblib.load(path)
    if artifact.get("training_policy", {}).get("sampling") != "none":
        raise ValueError(f"{path.name}: sampled artifact는 허용하지 않습니다.")
    print(path.name, artifact["model_id"], artifact["training_rows"], artifact["training_events"])
    files.download(str(path))
